In [ ]:
# Set up and Data file reading

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, year, month, sum as _sum, count
from pyspark.sql.types import DoubleType

spark = SparkSession.builder.appName("HealthcareClaims").getOrCreate()

df = spark.read.csv("dbfs:/FileStore/data/synthetic_claims.csv", header=True, inferSchema=True)
df = df.withColumn("service_date", to_date(col("service_date")))
df = df.withColumn("claim_amount", col("claim_amount").cast(DoubleType()))
df = df.withColumn("paid_amount", col("paid_amount").cast(DoubleType()))

print("Raw data preview:")
df.show(10)

In [ ]:
#Data Quality Checks


print("Null counts:")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# Basic validation
cleaned_df = df.filter(col("claim_amount") > 0).filter(col("patient_id").isNotNull())
print(f"Records after cleaning: {cleaned_df.count()}")

In [ ]:
#Transformations & Aggregations

summary_df = cleaned_df.groupBy(year("service_date").alias("year"), "diagnosis_code") \
    .agg(
        _sum("claim_amount").alias("total_claimed"),
        _sum("paid_amount").alias("total_paid"),
        count("*").alias("claim_count")
    ).orderBy("year", "total_claimed", ascending=False)

summary_df.show(20)

In [ ]:
#Write the data into data lake

summary_df.write.format("delta").mode("overwrite").save("/delta/claims_summary")

# Read back to verify
delta_df = spark.read.format("delta").load("/delta/claims_summary")
delta_df.show()

In [ ]:
# Final provider level summary
provider_summary = cleaned_df.groupBy("provider_id") \
    .agg(
        _sum("claim_amount").alias("total_billed"),
        count("*").alias("claims_submitted")
    ).orderBy("total_billed", ascending=False)

provider_summary.show()